# Tutorial 3: Cloud Detector Step-by-Step

Walkthrough of the internal mechanisms of the cloud detector.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from panoseti_analysis.io.models import load_classifier
from panoseti_analysis.paths import MODELS

## Load Model

In [ ]:
model, bundle = load_classifier(MODELS / "cloud_detector_v1.pt")
model.eval()
print("Model architecture:", model)

## Feature Extraction (Hann Window + FFT)

In [ ]:
ds_l2 = xr.open_zarr("results_tutorial/L2/obs_TEST.cloud.module_1.zarr", consolidated=False)
display(ds_l2)
plt.imshow(np.log(ds_l2["feature_deriv_fft"][0] + 1e-7))

In [ ]:
ds_l1 = xr.open_zarr("results_tutorial/L1/obs_TEST.dp_img16.module_1.L1.zarr", consolidated=False)
img = ds_l1["median_subtracted"].values

# Get a single frame
frame = img[0]
hann_2d = np.outer(np.hanning(32), np.hanning(32))

fft = np.abs(np.fft.fftn(frame * hann_2d))
fft = np.log(np.fft.fftshift(fft))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(frame)
ax1.set_title("Original")
ax2.imshow(fft)
ax2.set_title("FFT")
plt.show()